# RT-DETR-R50vd & RT-DETR-R101vd Training on Kaggle
## Official Peking University Pre-trained Weights (Hugging Face Transformers)
**Authors:** Phạm Hoàng Minh Châu, Đỗ Trung Quân, Nguyễn Nhật Thông | **Mentor:** Đặng Ngọc Minh Đức

### 📌 Pre-trained Model Hub:
- **RT-DETR-R50vd**: [`PekingU/rtdetr_r50vd`](https://huggingface.co/PekingU/rtdetr_r50vd) (ResNet-50-vd backbone, COCO pre-trained)
- **RT-DETR-R101vd**: [`PekingU/rtdetr_r101vd`](https://huggingface.co/PekingU/rtdetr_r101vd) (ResNet-101-vd backbone, COCO pre-trained)

### ⚙️ Kaggle Settings (Right Sidebar):
1. **Accelerator**: Chọn **GPU T4 x 2** (hoặc GPU P100).
2. **Internet**: Bật sang **ON** (để tải weights từ Hugging Face).
3. **Camber Secret**: Thêm `CAMBER_API_KEY` vào Kaggle Secrets; dataset được tải từ Camber stash, không cần Kaggle Dataset Input.

In [ ]:
import os, shutil, subprocess, time
from pathlib import Path
import torch
import pandas as pd
from datetime import datetime

# Fresh experiment namespace for the 2026-11-01 dataset. Old experiment checkpoints are NEVER loaded.
# Resume is supported ONLY within this new experiment, to protect results.csv on Kaggle restart.
EXPERIMENT_ID = "finding_nemo_human_reviewed_6cls_20261101"
# 1. Cấu hình Thư mục Local
W = Path("/kaggle/working")
ROOT = W / "rtdetr_training" / EXPERIMENT_ID
ROOT.mkdir(parents=True, exist_ok=True)
CSV_COLUMNS = "epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2\n"

# 2. Xác thực và Cài đặt Camber CLI
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("CAMBER_API_KEY")
except Exception:
    token = os.environ.get("CAMBER_API_KEY", "")

if not token: 
    raise RuntimeError("Thiếu CAMBER_API_KEY trong Kaggle Secrets")
os.environ["CAMBER_API_KEY"] = token

if shutil.which("camber") is None:
    print("Đang cài đặt Camber CLI...")
    q = subprocess.run("curl -sL https://cli.cambercloud.com/install-v2.sh | bash", shell=True, text=True, capture_output=True)
    if q.returncode != 0: raise RuntimeError(q.stderr)

os.environ["PATH"] = f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:" + os.environ.get("PATH", "")
CAMBER = shutil.which("camber") or str(Path.home() / ".camber/bin/camber")

# 3. Cấu hình Remote Camber
BASE = f"stash://bunpmc/projects/finding_nemo/rtdetr_training/{EXPERIMENT_ID}"
REMOTE_CKPT = f"{BASE}/checkpoints/"
REMOTE_STATE = f"{BASE}/state/"

def camber_cp(src, dst, required=False):
    r = subprocess.run([CAMBER, "stash", "cp", str(src), str(dst)], text=True, capture_output=True)
    if r.returncode != 0 and required:
        print(f"⚠️ Lỗi Camber Upload: {r.stderr}")
    return r.returncode == 0

print("Đang khởi tạo thư mục trên Camber...")
for d in [REMOTE_CKPT, REMOTE_STATE]:
    subprocess.run([CAMBER, "stash", "mkdir", d], text=True, capture_output=True)
print("✅ Hoàn tất cấu hình Camber!")

In [ ]:
# Cell 1: Environment & Package Installation
import os, sys, time, json
from pathlib import Path
import torch

print("=== Kaggle Environment Verification ===")
print("Python Version :", sys.version.split()[0])
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

num_gpus = torch.cuda.device_count()
print(f"Detected {num_gpus} GPU(s):")
for i in range(num_gpus):
    total_mem_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  [{i}] {torch.cuda.get_device_name(i)} ({total_mem_gb:.1f} GB VRAM)")

# Install required libraries for Hugging Face RT-DETR
!pip install -q --upgrade transformers accelerate pycocotools

# === EXECUTION SWITCHES (Rất hữu ích khi Save Version) ===
# Bật/tắt model muốn chạy trong phiên này (tránh vượt trần 12h của Kaggle):
TRAIN_R50 = False    # RT-DETR-R50vd (~5 tiếng trên 2x T4)
TRAIN_R101 = True   # RT-DETR-R101vd (~8 tiếng trên 2x T4)

# Cell 2: Safe Environment Setup & Fail-Fast Import Test
import os
import torch

# 1. Tắt TorchDynamo (nguyên nhân gây lỗi biên dịch kernel)
os.environ["TORCH_DYNAMO_DISABLE"] = "1"

# 2. Đánh lừa thư viện transformers để không gọi các C++ backend bị lỗi trên Kaggle
try:
    import transformers.utils.import_utils as t_utils
    t_utils.is_torch_distributed_available = lambda: False
    t_utils.is_torch_dynamo_available = lambda: False
    t_utils.is_torch_compile_available = lambda: False
    t_utils.is_torch_flex_attn_available = lambda: False
    t_utils.is_torchvision_available = lambda: False
    t_utils._torchvision_available = False
except Exception as e:
    print(f"Warning patching transformers utils: {e}")

# 3. Test import ngay lập tức (Fail Fast)
try:
    from transformers import RTDetrForObjectDetection, RTDetrConfig, RTDetrImageProcessor
    print("✅ ĐÃ NẠP THÀNH CÔNG RT-DETR TỪ HUGGING FACE!")
except Exception as e:
    print(f"❌ Import thất bại! Lỗi gốc: {e}")
    raise e # Dừng notebook ngay tại đây nếu có lỗi, không chạy tiếp mất thời gian

In [ ]:
# Cell 2: Download final human-reviewed train/val/test dataset from Camber
import hashlib
import os
import shutil
import subprocess
import tarfile
from pathlib import Path

DATASET_STASH = "stash://bunpmc/projects/finding_nemo/finding_nemo_human_reviewed_6cls_20261101.tar.gz"
ARCHIVE = Path("/kaggle/working/finding_nemo_human_reviewed_6cls_20261101.tar.gz")
ARCHIVE_SHA256 = "5555fd783a52551225cec93b981e4362dcba99637fb54f463480df9e9eaf1643"
EXTRACT_ROOT = Path("/tmp/camber_finding_nemo_6cls_20261101_extract")
dest_dataset_dir = Path("/tmp/finding_nemo_human_reviewed_6cls_20261101")
EXPECTED = {
    "train": {"images": 17903, "boxes": 25811},
    "val": {"images": 2271, "boxes": 3204},
    "test": {"images": 1931, "boxes": 2850},
}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def file_sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


# CAMBER and CAMBER_API_KEY are configured in the previous setup cell.
if not ARCHIVE.is_file() or file_sha256(ARCHIVE) != ARCHIVE_SHA256:
    if ARCHIVE.exists():
        print("Removing stale/incomplete archive:", ARCHIVE)
        ARCHIVE.unlink()
    print(f"Downloading dataset from {DATASET_STASH}")
    result = subprocess.run([CAMBER, "stash", "cp", DATASET_STASH, str(ARCHIVE)], check=False)
    if result.returncode != 0:
        raise RuntimeError("Camber download failed. Check CAMBER_API_KEY and stash access.")

actual_sha256 = file_sha256(ARCHIVE)
if actual_sha256 != ARCHIVE_SHA256:
    raise RuntimeError(f"Dataset checksum mismatch: expected {ARCHIVE_SHA256}, got {actual_sha256}")
print(f"Archive verified: {ARCHIVE} ({ARCHIVE.stat().st_size / (1024 ** 3):.2f} GiB)")

with ARCHIVE.open("rb") as handle:
    magic = handle.read(4)
if magic[:2] != b"\x1f\x8b":
    raise ValueError(f"Expected gzip archive, got header {magic.hex()}")

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
marker = EXTRACT_ROOT / ".extraction_complete"
if not marker.exists() or marker.read_text() != ARCHIVE_SHA256:
    if any(EXTRACT_ROOT.iterdir()):
        shutil.rmtree(EXTRACT_ROOT)
        EXTRACT_ROOT.mkdir(parents=True)
    print("Extracting verified .tar.gz ...")
    with tarfile.open(ARCHIVE, mode="r|gz") as archive:
        safe_root = EXTRACT_ROOT.resolve()
        for member in archive:
            target = (safe_root / member.name).resolve()
            if (not target.is_relative_to(safe_root) or member.issym() or member.islnk()
                    or not (member.isfile() or member.isdir())):
                raise ValueError(f"Unsafe archive member: {member.name}")
            archive.extract(member, path=safe_root, filter="data")
    marker.write_text(ARCHIVE_SHA256)
else:
    print("Reusing checksum-verified extraction")


def is_dataset_root(path):
    return all((path / "images" / split).is_dir() and (path / "labels" / split).is_dir()
               for split in EXPECTED)


roots = [path for path in [EXTRACT_ROOT, *EXTRACT_ROOT.rglob("*")]
         if path.is_dir() and is_dataset_root(path)]
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one complete train/val/test dataset root, found: {roots}")
source_root = roots[0]
print("Dataset root:", source_root)


def validate_split(root, split):
    image_dir = root / "images" / split
    label_dir = root / "labels" / split
    images = sorted(path for path in image_dir.iterdir()
                    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
    labels = {path.stem: path for path in label_dir.glob("*.txt")}
    image_stems = {path.stem for path in images}
    if image_stems != set(labels):
        raise RuntimeError(
            f"{split} image/label mismatch: images={len(image_stems)}, labels={len(labels)}"
        )
    boxes = sum(sum(1 for line in labels[image.stem].read_text().splitlines() if line.strip())
                for image in images)
    expected = EXPECTED[split]
    if len(images) != expected["images"] or boxes != expected["boxes"]:
        raise RuntimeError(
            f"{split} mismatch: got {len(images)} images/{boxes} boxes; "
            f"expected {expected['images']}/{expected['boxes']}"
        )
    print(f"Verified {split}: {len(images):,} images, {boxes:,} boxes")
    return {path.name for path in images}


names = {split: validate_split(source_root, split) for split in EXPECTED}
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = names[left] & names[right]
    if overlap:
        raise RuntimeError(f"{left}/{right} share {len(overlap)} filenames: {sorted(overlap)[:3]}")

# Create a unified stable path used by the remaining notebook cells.
dest_dataset_dir.mkdir(parents=True, exist_ok=True)
for kind in ("images", "labels"):
    (dest_dataset_dir / kind).mkdir(exist_ok=True)
    for split in EXPECTED:
        target = dest_dataset_dir / kind / split
        expected_source = (source_root / kind / split).resolve()
        if target.is_symlink():
            if target.resolve() == expected_source:
                continue
            target.unlink()
        if target.exists():
            raise RuntimeError(f"Refusing to overwrite {target}")
        target.symlink_to(expected_source, target_is_directory=True)

print(f"Dataset ready: {dest_dataset_dir}")
for split in EXPECTED:
    print(split, "images:", len(list((dest_dataset_dir / "images" / split).iterdir())))


In [ ]:
# Cell 3: Automatic YOLO-to-COCO JSON Annotation Generator (Runs in ~5 seconds!)
from PIL import Image
from tqdm import tqdm

CLASS_NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand"
]

ann_dir = dest_dataset_dir / "annotations"
ann_dir.mkdir(parents=True, exist_ok=True)

def convert_yolo_to_coco(split_name: str):
    out_json = ann_dir / f"{split_name}.json"
    # Rebuild annotation JSON to prevent reuse from a previous dataset version.
    
    print(f"Đang tạo COCO JSON cho split '{split_name}'...")
    img_dir = dest_dataset_dir / "images" / split_name
    lbl_dir = dest_dataset_dir / "labels" / split_name
    
    images_list = []
    annotations_list = []
    categories = [{"id": i, "name": name} for i, name in enumerate(CLASS_NAMES)]
    
    ann_id = 0
    img_files = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png")))
    
    for img_id, img_path in enumerate(tqdm(img_files, desc=f"Converting {split_name}")):
        with Image.open(img_path) as im:
            w, h = im.size
            
        images_list.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": w,
            "height": h
        })
        
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            with open(lbl_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        xc, yc, bw, bh = [float(x) for x in parts[1:5]]
                        # Convert normalized [xc, yc, w, h] to COCO [xmin, ymin, w, h]
                        xmin = (xc - bw / 2.0) * w
                        ymin = (yc - bh / 2.0) * h
                        box_w = bw * w
                        box_h = bh * h
                        area = box_w * box_h
                        
                        annotations_list.append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": cls_id,
                            "bbox": [round(xmin, 2), round(ymin, 2), round(box_w, 2), round(box_h, 2)],
                            "area": round(area, 2),
                            "iscrowd": 0
                        })
                        ann_id += 1
                        
    coco_dict = {
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories
    }
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(coco_dict, f)
    print(f"✅ Tạo xong {out_json.name}: {len(images_list):,} ảnh, {len(annotations_list):,} nhãn!")
    return out_json

train_json = convert_yolo_to_coco("train")
val_json = convert_yolo_to_coco("val")

In [ ]:
# Cell 4: PyTorch Dataset & DataLoader for Hugging Face RT-DETR
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# Bộ tiền xử lý PyTorch thuần (Chính xác 100% theo chuẩn PekingU RT-DETR, không lo lỗi backend)
class PowerlineImageProcessor:
    def __init__(self, size=(640, 640)):
        self.size = size
        
    @classmethod
    def from_pretrained(cls, *args, **kwargs):
        return cls(size=(640, 640))
        
    def save_pretrained(self, save_dir):
        import json
        from pathlib import Path
        config = {
            "do_normalize": False,
            "do_rescale": True,
            "do_resize": True,
            "image_processor_type": "RTDetrImageProcessor",
            "rescale_factor": 1.0 / 255.0,
            "size": {"height": self.size[0], "width": self.size[1]}
        }
        p = Path(save_dir) / "preprocessor_config.json"
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)
            
    def __call__(self, images, return_tensors="pt"):
        if not isinstance(images, list):
            images = [images]
        tensors = []
        for img in images:
            img_rgb = img.convert("RGB").resize(self.size, Image.BILINEAR)
            arr = np.array(img_rgb, dtype=np.float32).transpose(2, 0, 1) / 255.0
            tensors.append(torch.from_numpy(arr))
        return {"pixel_values": torch.stack(tensors) if return_tensors == "pt" else tensors}

try:
    from transformers import RTDetrImageProcessor
    print("✅ Successfully loaded official Hugging Face RTDetrImageProcessor")
except Exception as e:
    print(f"⚠️ Hugging Face image backend unavailable ({e}). Đang sử dụng Native High-Speed PyTorch Processor chuẩn xác 100%!")
    RTDetrImageProcessor = PowerlineImageProcessor

class PowerlineCOCODataset(Dataset):
    def __init__(self, img_dir, json_path, processor):
        self.img_dir = Path(img_dir)
        self.processor = processor
        with open(json_path, "r", encoding="utf-8") as f:
            coco = json.load(f)
            
        self.images = {img["id"]: img for img in coco["images"]}
        self.img_ids = list(self.images.keys())
        
        # Gom nhóm annotations theo image_id
        self.img_to_anns = {i: [] for i in self.img_ids}
        for ann in coco["annotations"]:
            self.img_to_anns[ann["image_id"]].append(ann)
            
    def __len__(self):
        return len(self.img_ids)
        
    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]
        img_path = self.img_dir / img_info["file_name"]
        
        image = Image.open(img_path).convert("RGB")
        w, h = img_info["width"], img_info["height"]
        
        anns = self.img_to_anns[img_id]
        boxes = []
        classes = []
        for ann in anns:
            xmin, ymin, bw, bh = ann["bbox"]
            xc = (xmin + bw / 2.0) / w
            yc = (ymin + bh / 2.0) / h
            nw = bw / w
            nh = bh / h
            boxes.append([xc, yc, nw, nh])
            classes.append(ann["category_id"])
            
        if not boxes:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            classes = torch.zeros((0,), dtype=torch.long)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            classes = torch.tensor(classes, dtype=torch.long)
            
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        
        target = {"class_labels": classes, "boxes": boxes}
        return pixel_values, target

def collate_fn(batch):
    pixel_values = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return pixel_values, targets

print("DataLoader components initialized successfully!")

In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import gc # Thêm thư viện này

# 1. Nâng cấp Dataset (Bổ sung image_id và orig_size cho COCO Eval)
class PowerlineCOCODataset(Dataset):
    def __init__(self, img_dir, json_path, processor):
        self.img_dir = Path(img_dir)
        self.processor = processor
        with open(json_path, "r", encoding="utf-8") as f:
            coco = json.load(f)
        self.images = {img["id"]: img for img in coco["images"]}
        self.img_ids = list(self.images.keys())
        self.img_to_anns = {i: [] for i in self.img_ids}
        for ann in coco["annotations"]:
            self.img_to_anns[ann["image_id"]].append(ann)
            
    def __len__(self): return len(self.img_ids)
        
    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]
        image = Image.open(self.img_dir / img_info["file_name"]).convert("RGB")
        w, h = img_info["width"], img_info["height"]
        
        boxes, classes = [], []
        for ann in self.img_to_anns[img_id]:
            xmin, ymin, bw, bh = ann["bbox"]
            # Convert COCO bbox to HF normalized [xc, yc, w, h]
            boxes.append([(xmin + bw/2.0)/w, (ymin + bh/2.0)/h, bw/w, bh/h])
            classes.append(ann["category_id"])
            
        if not boxes:
            boxes, classes = torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.long)
        else:
            boxes, classes = torch.tensor(boxes, dtype=torch.float32), torch.tensor(classes, dtype=torch.long)
            
        pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        target = {"class_labels": classes, "boxes": boxes, "orig_size": torch.tensor([h, w]), "image_id": torch.tensor([img_id])}
        return pixel_values, target

def collate_fn(batch):
    pixel_values = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return pixel_values, targets

def safe_device_property(self):
    try:
        return next(self.parameters()).device
    except StopIteration:
        return torch.device(f"cuda:{torch.cuda.current_device()}")

RTDetrForObjectDetection.device = property(safe_device_property)

# 2. Nâng cấp Custom DataParallel (Thu thập Loss chuẩn xác)
class RTDetrDataParallel(nn.DataParallel):
    def scatter(self, inputs, kwargs, device_ids):
        pixel_values = kwargs.get("pixel_values", inputs[0] if len(inputs) > 0 else None)
        scattered_pixels = nn.parallel.scatter(pixel_values, device_ids)
        labels = kwargs.get("labels", None)
        scattered_labels = []
        if labels is not None:
            current_idx = 0
            for px, dev in zip(scattered_pixels, device_ids):
                chunk_len = px.size(0)
                chunk = labels[current_idx : current_idx + chunk_len]
                current_idx += chunk_len
                device = torch.device(f"cuda:{dev}")
                moved_chunk = [{"class_labels": t["class_labels"].to(device), "boxes": t["boxes"].to(device)} for t in chunk]
                scattered_labels.append(moved_chunk)
        else:
            scattered_labels = [None] * len(scattered_pixels)
            
        scat_kwargs = tuple({"pixel_values": px, "labels": lbl} for px, lbl in zip(scattered_pixels, scattered_labels))
        return tuple(tuple() for _ in scattered_pixels), scat_kwargs

    def forward(self, *inputs, **kwargs):
        if not self.device_ids: return self.module(*inputs, **kwargs)
        inputs, kwargs = self.scatter(inputs, kwargs, self.device_ids)
        replicas = self.replicate(self.module, self.device_ids[:len(inputs)])
        outputs = self.parallel_apply(replicas, inputs, kwargs)
        
        losses, loss_ce, loss_box = [], [], []
        for out in outputs:
            if hasattr(out, 'loss') and out.loss is not None:
                losses.append(out.loss.unsqueeze(0))
                loss_ce.append(out.loss_dict.get('loss_ce', torch.tensor(0.0, device=out.loss.device)).unsqueeze(0))
                # Gộp L1 Bbox và GIoU
                total_box = out.loss_dict.get('loss_bbox', torch.tensor(0.0)) + out.loss_dict.get('loss_giou', torch.tensor(0.0))
                loss_box.append(total_box.to(out.loss.device).unsqueeze(0))
                
        if losses:
            # Tạo MockOutput mô phỏng chuẩn Object của HuggingFace
            class MockOutput:
                def __init__(self, l, l_ce, l_box):
                    self.loss = l
                    self.loss_dict = {'loss_ce': l_ce, 'loss_bbox': l_box}
            return MockOutput(
                torch.cat([l.to(self.output_device) for l in losses]).mean(),
                torch.cat([l.to(self.output_device) for l in loss_ce]).mean(),
                torch.cat([l.to(self.output_device) for l in loss_box]).mean()
            )
        return self.gather(outputs, self.output_device)

import time
import torch.optim as optim
from tqdm import tqdm
from transformers import RTDetrImageProcessor, RTDetrForObjectDetection, RTDetrConfig

# Đảm bảo các biến dùng chung được khai báo lại rõ ràng
CLASS_NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
id2label = {i: name for i, name in enumerate(CLASS_NAMES)}
label2id = {name: i for i, name in enumerate(CLASS_NAMES)}

def train_peking_rtdetr(model_variant: str = "r50vd", epochs: int = 50, batch_size: int = 16, accum_steps: int = 1, resume: bool = True):
    repo_id = f"PekingU/rtdetr_{model_variant}"
    model_root = ROOT / model_variant
    weights_dir = model_root / "weights"
    model_root.mkdir(parents=True, exist_ok=True)
    weights_dir.mkdir(parents=True, exist_ok=True)
    csv_path = model_root / "result.csv"
    remote_model = f"{BASE}/{model_variant}"
    remote_ckpt = f"{remote_model}/checkpoints"
    remote_state = f"{remote_model}/state"
    for remote in (remote_model, remote_ckpt, remote_state):
        subprocess.run([CAMBER, "stash", "mkdir", remote], capture_output=True, text=True)

    processor = RTDetrImageProcessor.from_pretrained(repo_id)
    
    train_ds = PowerlineCOCODataset(dest_dataset_dir / "images" / "train", train_json, processor)
    val_ds = PowerlineCOCODataset(dest_dataset_dir / "images" / "val", val_json, processor)
    
    # Đã áp dụng các tối ưu chống tràn RAM (giảm workers, tắt persistent)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn, pin_memory=True, prefetch_factor=2)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, collate_fn=collate_fn, pin_memory=True, prefetch_factor=2)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = RTDetrForObjectDetection.from_pretrained(repo_id, num_labels=len(CLASS_NAMES), id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True).to(device)
    
    if torch.cuda.device_count() > 1:
        model = RTDetrDataParallel(model)
        
    optimizer = optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad], "lr": 1e-5},
        {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad], "lr": 2e-4}
    ], weight_decay=1e-4)
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    
    coco_gt = COCO(val_json)
    
    # Resume ONLY this new dataset experiment, never an older checkpoint.
    # Resume must use an epoch-matched FULL state checkpoint (optimizer/scheduler/AMP).
    # Never infer an epoch from an unrelated CSV or silently load model-only weights.
    start_epoch = 1
    best_map50_95 = 0.0
    full_state_path = weights_dir / "training_state.pt"
    remote_full_state = f"{remote_state}/training_state.pt"
    if resume:
        if not full_state_path.exists():
            camber_cp(remote_full_state, full_state_path, required=False)
        if full_state_path.exists():
            saved = torch.load(full_state_path, map_location=device, weights_only=False)
            if saved.get("model_variant") != model_variant or saved.get("experiment_id") != EXPERIMENT_ID:
                raise RuntimeError("Checkpoint model/dataset does not match this training run")
            target = model.module if hasattr(model, "module") else model
            target.load_state_dict(saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            lr_scheduler.load_state_dict(saved["scheduler"])
            scaler.load_state_dict(saved["scaler"])
            start_epoch = int(saved["epoch"]) + 1
            best_map50_95 = float(saved.get("best_map50_95", 0))
            print(f"Resuming {model_variant} at epoch {start_epoch}")
            if not csv_path.exists():
                camber_cp(f"{remote_state}/result.csv", csv_path, required=False)
            if not csv_path.exists():
                raise RuntimeError("Checkpoint found but result.csv unavailable; refusing to lose training history")
            df = pd.read_csv(csv_path)
            if df.empty or int(df["epoch"].max()) < start_epoch - 1:
                raise RuntimeError("CSV log is behind checkpoint. Recover original log before resuming")
            if start_epoch > epochs:
                print("All requested epochs completed")
                return best_map50_95
        else:
            # A CSV without checkpoint cannot safely resume weights/optimizer state.
            if not csv_path.exists():
                camber_cp(f"{remote_state}/result.csv", csv_path, required=False)
            if csv_path.exists() and len(pd.read_csv(csv_path)) > 0:
                raise RuntimeError("Found training history but no full-state checkpoint. Refusing to overwrite/resume incorrectly")

    if not csv_path.exists():
        csv_path.write_text(CSV_COLUMNS)

    for epoch in range(start_epoch, epochs + 1):
        epoch_start_time = time.time()
        model.train()
        train_box_loss, train_cls_loss = 0.0, 0.0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        optimizer.zero_grad()
        for batch_idx, (pixel_values, targets) in enumerate(train_pbar):
            pixel_values = pixel_values.to(device)
            formatted_targets = [{"class_labels": t["class_labels"].to(device), "boxes": t["boxes"].to(device)} for t in targets]
            
            with torch.amp.autocast("cuda"):
                outputs = model(pixel_values=pixel_values, labels=formatted_targets)
                loss = outputs.loss / accum_steps
                
            scaler.scale(loss).backward()
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            t_box = outputs.loss_dict.get('loss_bbox', torch.tensor(0.0)) + outputs.loss_dict.get('loss_giou', torch.tensor(0.0))
            train_box_loss += t_box.item() if isinstance(t_box, torch.Tensor) else t_box
            
            t_cls = outputs.loss_dict.get('loss_ce', torch.tensor(0.0))
            train_cls_loss += t_cls.item() if isinstance(t_cls, torch.Tensor) else t_cls
            
            train_pbar.set_postfix({"loss": f"{loss.item() * accum_steps:.4f}"})
            
            del pixel_values, targets, formatted_targets, outputs, loss
            
        avg_t_box = train_box_loss / len(train_loader)
        avg_t_cls = train_cls_loss / len(train_loader)
        lr_pg0 = optimizer.param_groups[0]["lr"]
        lr_pg1 = optimizer.param_groups[1]["lr"]
        
        # Chỉ step scheduler sau khi đã hoàn thành epoch (tránh step đúp nếu đã tua nhanh)
        if epoch >= start_epoch:
            lr_scheduler.step()
        
        # ================= VALIDATION & COCO EVAL =================
        eval_model = model.module if hasattr(model, 'module') else model
        eval_model.eval()
        val_box_loss, val_cls_loss = 0.0, 0.0
        coco_results = []
        
        with torch.no_grad():
            for pixel_values, targets in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                pixel_values = pixel_values.to(device)
                formatted_targets = [{"class_labels": t["class_labels"].to(device), "boxes": t["boxes"].to(device)} for t in targets]
                
                with torch.amp.autocast("cuda"):
                    outputs = eval_model(pixel_values=pixel_values, labels=formatted_targets)
                    
                    # Dùng .get() giống hệt vòng lặp Train để tránh KeyError
                    v_box = outputs.loss_dict.get('loss_bbox', torch.tensor(0.0)) + outputs.loss_dict.get('loss_giou', torch.tensor(0.0))
                    val_box_loss += v_box.item() if isinstance(v_box, torch.Tensor) else v_box
                    
                    v_cls = outputs.loss_dict.get('loss_ce', torch.tensor(0.0))
                    val_cls_loss += v_cls.item() if isinstance(v_cls, torch.Tensor) else v_cls
                    
                orig_sizes = torch.stack([t["orig_size"] for t in targets]).to(device)
                img_ids = [t["image_id"].item() for t in targets]
                results = processor.post_process_object_detection(outputs, target_sizes=orig_sizes, threshold=0.001)
                
                for img_id, res in zip(img_ids, results):
                    scores = res["scores"].cpu().tolist()
                    labels = res["labels"].cpu().tolist()
                    boxes = res["boxes"].cpu().tolist()
                    for score, label, box in zip(scores, labels, boxes):
                        xmin, ymin, xmax, ymax = box
                        coco_results.append({
                            "image_id": img_id,
                            "category_id": label,
                            "bbox": [xmin, ymin, xmax - xmin, ymax - ymin],
                            "score": score
                        })
                del pixel_values, targets, formatted_targets, outputs, orig_sizes, results
        
        avg_v_box = val_box_loss / len(val_loader)
        avg_v_cls = val_cls_loss / len(val_loader)
        
        map50, map50_95, precision, recall = 0.0, 0.0, 0.0, 0.0
        if len(coco_results) > 0:
            res_json = ROOT / "temp_res.json"
            with open(res_json, "w") as f: json.dump(coco_results, f)
            try:
                coco_dt = coco_gt.loadRes(str(res_json))
                coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
                coco_eval.evaluate()
                coco_eval.accumulate()
                coco_eval.summarize()
                map50_95 = coco_eval.stats[0]
                map50 = coco_eval.stats[1]
                recall = coco_eval.stats[8] 
                precision = coco_eval.stats[0]
            except Exception as e:
                print(f"⚠️ Lỗi Evaluator (Epoch {epoch}): {e}")

        epoch_time = (time.time() - epoch_start_time) / 3600
        
        # ================= LƯU LOG VÀ ĐỒNG BỘ CAMBER =================
        log_line = f"{epoch},{epoch_time:.4f},{avg_t_box:.4f},{avg_t_cls:.4f},0.0,{precision:.4f},{recall:.4f},{map50:.4f},{map50_95:.4f},{avg_v_box:.4f},{avg_v_cls:.4f},0.0,{lr_pg0:.6e},{lr_pg1:.6e},0.0\n"
        with open(csv_path, "a") as f: f.write(log_line)
        # Append-only per-epoch CSV snapshot, then update latest remote log.
        csv_snapshot = model_root / f"result_epoch{epoch:03d}.csv"
        shutil.copy2(csv_path, csv_snapshot)
        if not camber_cp(csv_snapshot, f"{remote_state}/{csv_snapshot.name}"):
            raise RuntimeError("CSV history snapshot upload failed; training stopped to protect logs")
        if not camber_cp(csv_path, f"{remote_state}/result.csv"):
            raise RuntimeError("CSV history upload failed; training stopped to protect logs")

        eval_model = model.module if hasattr(model, "module") else model
        torch.save(eval_model.state_dict(), weights_dir / "last.pt")
        if epoch % 5 == 0 or epoch == 1:
            pt_name = f"epoch{epoch}.pt"
            torch.save(eval_model.state_dict(), weights_dir / pt_name)
            camber_cp(weights_dir / pt_name, f"{remote_ckpt}/{pt_name}")
        if map50_95 > best_map50_95:
            best_map50_95 = map50_95
            torch.save(eval_model.state_dict(), weights_dir / "best.pt")
            camber_cp(weights_dir / "best.pt", f"{remote_ckpt}/best.pt")

        # Full checkpoint is uploaded only AFTER corresponding CSV was saved.
        state = {
            "epoch": epoch, "experiment_id": EXPERIMENT_ID,
            "model_variant": model_variant, "model": eval_model.state_dict(),
            "optimizer": optimizer.state_dict(), "scheduler": lr_scheduler.state_dict(),
            "scaler": scaler.state_dict(), "best_map50_95": float(best_map50_95)
        }
        temp_state = weights_dir / "training_state.tmp.pt"
        torch.save(state, temp_state)
        temp_state.replace(full_state_path)
        if not camber_cp(full_state_path, remote_full_state):
            raise RuntimeError("Checkpoint upload failed; local checkpoint preserved, stop training")
        # Giải phóng bộ nhớ trước epoch mới
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
# Cell 6: Train RT-DETR-R50vd (Peking University Pre-trained)
# Batch size = 8, Accumulation Steps = 2 -> Effective Batch Size = 16
if TRAIN_R50:
    train_peking_rtdetr(model_variant="r50vd", epochs=50, batch_size=32, accum_steps=1)
else:
    print("⏩ Bỏ qua RT-DETR-R50vd (TRAIN_R50 = False)")

In [ ]:
# Cell 7: Train RT-DETR-R101vd (Peking University Pre-trained)
# Batch size = 4, Accumulation Steps = 4 -> Effective Batch Size = 16
if TRAIN_R101:
    train_peking_rtdetr(model_variant="r101vd", epochs=50, batch_size=16, accum_steps=1)
else:
    print("⏩ Bỏ qua RT-DETR-R101vd (TRAIN_R101 = False)")

In [ ]:
# Cell 8: Zip Checkpoints and Logs for One-Click Download
import shutil
from pathlib import Path

# Sửa lại đúng thư mục ROOT mà chúng ta đã setup ở đầu notebook
runs_dir = Path("/kaggle/working/rtdetr_training") 
if runs_dir.exists():
    print("Packaging weights and results...")
    zip_path = shutil.make_archive("/kaggle/working/peking_rtdetr_6cls_20261101_checkpoints", "zip", runs_dir)
    print(f"Successfully created: {zip_path}")
    print(f"Size: {Path(zip_path).stat().st_size / (1024**2):.1f} MB")
    print("Tải file 'peking_rtdetr_checkpoints.zip' từ mục Output về máy local!")
else:
    print("Chưa có thư mục rtdetr_training. Hãy chạy train trước.")